In [1]:
import os
import json
import sys
import argparse
import datetime
import pandas as pd
import time
import traceback
from trade_core import InitCore
# from exchange_plugin.common_info import InitCommonInfo
# from monitor_engine.kimp_core_monitor import InitKimpCoreMonitor
# from etc.db_handler.create_schema_tables import InitDBClient
# from trigger_engine.snatcher import InitSnatcher
from etc.register_monitor_msg import RegisterMonitorMsg
from etc.redis_connector.redis_connector import InitRedis
import _pickle as pickle

# for jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [ ]:
redis_cli = InitRedis()

In [ ]:
temp = [x.decode('utf-8') for x in redis_cli.redis_conn.keys()]
temp = [x for x in temp if 'SERVER_CHECK' in x]
temp

In [2]:
logging_dir = "/home/trade_core/"
with open("/home/trade_core/trade_core_config.json") as f:
    config = json.load(f)
proc_n = 1
# node = config['node']
node = 'trade_core1'
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id = config['telegram_admin_id']['charlie1155']
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
enabled_markets = config['node_settings'][node]['enabled_markets']

# db dict
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

my_okx_demo_api_key = "bbb8a09a-6ea2-4686-add5-1095c918b7f4"
my_okx_demo_secret_key = "FBEAD86057449BAEC3FFA8A80CE76E4F"
my_okx_demo_passphrase = "121431Rn!!"

In [3]:
core = InitCore(logging_dir, proc_n, node, admin_id, register_monitor_msg, exchange_api_key_dict, enabled_markets, db_dict)

InitDBClient|SCHEMA: trade_core already exists.
InitDBClient|TABLE: user_info already exists.
InitDBClient|TABLE: exchange_config already exists.
InitDBClient|TABLE: trade already exists.


2023-12-27 16:56:58,022 - trade_core - INFO - InitCore|server_check_status_list has been initiated.
2023-12-27 16:56:58,025 - trade_core - INFO - InitCore|InitCore initiated with proc_n=1


2023-12-27 16:56:58,590 - okx_plug - INFO - okx_plug_logger started.
2023-12-27 16:56:58,954 - upbit_plug - INFO - upbit_plug_logger started.
2023-12-27 16:56:58,969 - update_dollar - INFO - fetch_dollar|Dollar price (1297.0 KRW) has been updated.
2023-12-27 16:56:59,116 - binance_plug - INFO - binance_plug_logger started.
2023-12-27 16:56:59,117 - bithumb_plug - INFO - bithumb_plug_logger started.
2023-12-27 16:56:59,119 - bybit_plug - INFO - bybit_plug_logger started.
2023-12-27 16:56:59,121 - trade_core - INFO - InitCore|update_binance_spot_ticker_df thread has started.
2023-12-27 16:56:59,122 - trade_core - INFO - InitCore|update_binance_spot_info_df thread has started.
2023-12-27 16:56:59,124 - trade_core - INFO - InitCore|update_binance_usd_m_ticker_df thread has started.
2023-12-27 16:56:59,126 - trade_core - INFO - InitCore|update_binance_usd_m_info_df thread has started.
2023-12-27 16:56:59,129 - trade_core - INFO - InitCore|update_upbit_spot_ticker_df thread has started.
2023

In [ ]:
core.server_check_status_list

In [ ]:
target_market_code = "UPBIT_SPOT/KRW"
origin_market_code = "BINANCE_USD_M/USDT"
core.get_premium_df(target_market_code, origin_market_code)

In [4]:
from etc.db_handler.postgres_client import InitDBClient as InitPostgresDBClient
from loggers.logger import KimpBotLogger
from threading import Thread
from multiprocessing import Process, Manager
from threading import Thread
from psycopg2 import extras

class InitTrigger:
    def __init__(self, admin_id, node, server_check_status_list, get_premium_df, enabled_markets, register_monitor_msg, remote_redis_client, db_dict, mongo_client, postgres_client, logging_dir):
        self.admin_id = admin_id
        self.node = node
        self.server_check_status_list = server_check_status_list
        self.get_premium_df = get_premium_df
        self.enabled_markets = enabled_markets
        self.register_monitor_msg = register_monitor_msg
        self.remote_redis_client = remote_redis_client
        self.db_dict = db_dict
        self.mongo_client = mongo_client
        self.postgres_client = postgres_client
        self.logging_dir = logging_dir
        self.logger = KimpBotLogger(logging_dir, "trigger").logger
        self.manager = Manager()
        self.user_info_dict = self.manager.dict()
        self.user_info_dict_initiated = False
        self.exchange_config_df_dict = self.manager.dict()
        self.exchange_config_df_dict_initiated = False
        self.trade_df_dict = self.manager.dict()
        self.load_user_info_thread = Thread(target=self.load_user_info, daemon=True)
        self.load_user_info_thread.start()
        self.load_exchange_config_thread = Thread(target=self.load_exchange_config, daemon=True)
        self.load_exchange_config_thread.start()
        for each_market_combi_code in self.enabled_markets:        
            self.start_trigger_loop_proc = Process(target=self.start_trigger_loop, args=(each_market_combi_code, self.trade_df_dict), daemon=True)
            self.start_trigger_loop_proc.start()
        # while self.user_info_dict_initiated is False or self.exchange_config_df_dict_initiated is False or [self.trade_df_dict.get(each_market_combi_code) for each_market_combi_code in self.enabled_markets].count(None) != 0:
        #     time.sleep(0.2)
        while self.user_info_dict_initiated is False:
            time.sleep(0.2)
        self.logger.info("user_info_df, exchange_config_df_dict, trade_df_dict have been initialized")
        time.sleep(20)

    def is_table_empty(self, conn, table_name):
        with conn.cursor() as cur:
            cur.execute(f"SELECT COUNT(*) FROM {table_name}")
            count = cur.fetchone()[0]
            return count == 0

    def get_column_names(self, conn, table_name):
        with conn.cursor() as cur:
            cur.execute("""
                SELECT column_name 
                FROM information_schema.columns 
                WHERE table_schema = 'public' AND table_name = %s
                ORDER BY ordinal_position;
            """, (table_name,))
            return [row[0] for row in cur.fetchall()]

    def load_user_info(self, table_name='user_info', loop_interval_secs=1):
        while True:
            try:
                conn = self.postgres_client.pool.getconn()
                curr = conn.cursor(cursor_factory=extras.RealDictCursor)
                curr.execute(f"SELECT * FROM {table_name}")
                fetched_dict_list = curr.fetchall()
                for fetched_dict in fetched_dict_list:
                    each_user_info_dict = {
                        "email": fetched_dict['email'],
                        "telegram_id": fetched_dict['telegram_id'],
                        "telegram_name": fetched_dict['telegram_name'],
                        "alarm_num": fetched_dict['alarm_num'],
                        "alarm_period": fetched_dict['alarm_period'],
                    }
                    self.user_info_dict[fetched_dict['user_uuid']] = each_user_info_dict
                self.postgres_client.pool.putconn(conn)
                if self.user_info_dict_initiated is False:
                    self.user_info_dict_initiated = True
                time.sleep(loop_interval_secs)
            except Exception as e:
                # rollback the transaction if any error while inserting
                self.postgres_client.pool.putconn(conn, close=True)
                self.logger.error(f"Error in load_user_info: {e}")
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"load_user_info", content=traceback.format_exc(), code=None, sent_switch=0, send_counts=1, remark=None)
                time.sleep(10)

    def load_exchange_config(self, table_name='exchange_config', loop_interval_secs=1):
        while True:
            try:
                # First check whether the table is empty
                conn = self.postgres_client.pool.getconn()
                curr = conn.cursor(cursor_factory=extras.RealDictCursor)
                if self.is_table_empty(conn, table_name):
                    # # Get the column names
                    # column_names = self.get_column_names(conn, table_name)
                    # # Create empty dataframe
                    # self.exchange_config_df = pd.DataFrame(columns=column_names)
                    pass
                else:
                    curr.execute(f"SELECT * FROM {table_name}")
                    exchange_config_df = pd.DataFrame(curr.fetchall())
                    target_market_code_unique = exchange_config_df['target_market_code'].unique()
                    origin_market_code_unique = exchange_config_df['origin_market_code'].unique()
                    for target_market_code in target_market_code_unique:
                        for origin_market_code in origin_market_code_unique:
                            market_combi_code = f"{target_market_code}:{origin_market_code}"
                            self.exchange_config_df_dict[market_combi_code] = exchange_config_df[(exchange_config_df['target_market_code'] == target_market_code) & (exchange_config_df['origin_market_code'] == origin_market_code)]
                self.postgres_client.pool.putconn(conn)
                if self.exchange_config_df_dict_initiated is False:
                    self.exchange_config_df_dict_initiated = True
                time.sleep(loop_interval_secs)
            except Exception as e:
                # rollback the transaction if any error while inserting
                self.postgres_client.pool.putconn(conn, close=True)
                self.logger.error(f"Error in load_exchange_config: {e}")
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"load_exchange_config", content=traceback.format_exc(), code=None, sent_switch=0, send_counts=1, remark=None)
                time.sleep(10)
        
    def start_trigger_loop(self, market_combi_code, trade_df_dict, table_name='trade', loop_interval_secs=5):
        # TEST
        target_market_code, origin_market_code = market_combi_code.split(':')
        self.logger.info(f"start_trigger_loop: {market_combi_code}")
        postgres_client = InitPostgresDBClient(**{**self.db_dict, 'database': 'trade_core'})
        self.logger.info(f"postgres client has been initiated for {market_combi_code}")
        while True:
            try:
                # First check whether the table is empty
                conn = postgres_client.pool.getconn()
                curr = conn.cursor(cursor_factory=extras.RealDictCursor)
                if self.is_table_empty(conn, table_name):
                    # Get the column names
                    column_names = self.get_column_names(conn, table_name)
                    # Create empty dataframe
                    trade_df_dict[market_combi_code] = pd.DataFrame(columns=column_names)
                else:
                    curr.execute(f"SELECT * FROM {table_name} WHERE target_market_code = %s AND origin_market_code = %s", (target_market_code, origin_market_code))
                    temp_trade_df = pd.DataFrame(curr.fetchall())
                    exchange_config_df = self.exchange_config_df_dict[market_combi_code][['user_uuid','service_datetime_end']]
                    valid_user_uuid_list = exchange_config_df[exchange_config_df['service_datetime_end']>=datetime.datetime.utcnow()]['user_uuid'].tolist()
                    trade_df_dict[market_combi_code] = temp_trade_df[temp_trade_df['user_uuid'].isin(valid_user_uuid_list)]
                    trade_df = trade_df_dict[market_combi_code]
                postgres_client.pool.putconn(conn)
                # Loading data done
                if len(trade_df) == 0:
                    time.sleep(loop_interval_secs)
                    continue

                trade_df = self.trade_df_dict[market_combi_code]
                premium_df = self.get_premium_df(*market_combi_code.split(':'))
                merged_df = trade_df.merge(premium_df, on='base_asset')
                merged_df['SL_premium_value'] = merged_df.apply(lambda x: x['SL_premium'] if x['usdt_conversion'] == False else x['SL_premium']*x['dollar'], axis=1)
                merged_df['LS_premium_value'] = merged_df.apply(lambda x: x['LS_premium'] if x['usdt_conversion'] == False else x['LS_premium']*x['dollar'], axis=1)

                # switch None: 최초, 0: 하향돌파 시, 1: 상향돌파 시
                # auto_trade_switch 0: 진입대기, -1: 탈출대기, 1:탈출완료, 2:탈출에러
                # case 1. switch None or False(0), High 돌파
                high_break_trade_df = (merged_df[((merged_df['trigger_switch'].isnull())|(merged_df['trigger_switch']==False))
                            &(merged_df['SL_premium']>=merged_df['high'])&(merged_df['trade_switch']!=3)])

                # case 2. switch None or True(1), Low 돌파
                low_break_trade_df = (merged_df[((merged_df['trigger_switch'].isnull())|(merged_df['trigger_switch']==True))
                            &(merged_df['LS_premium']<=merged_df['low'])&(merged_df['trade_switch']!=3)])

                # trade_swtich == 0: 진입대기, -1: 탈출대기, 1:탈출완료, 2:탈출에러, 3:거래 진행 중
                if len(high_break_trade_df) != 0:
                    high_break_trade_uuid_list = high_break_trade_df['uuid'].to_list()
                    # UPDATE database
                    conn = postgres_client.pool.getconn()
                    curr = conn.cursor()
                    curr.execute(f"UPDATE trade SET trigger_switch = 1 WHERE uuid IN %s", (tuple(high_break_trade_uuid_list),))
                    conn.commit()
                    postgres_client.pool.putconn(conn)
                    self.high_break(high_break_trade_df)

                if len(low_break_trade_df) != 0:
                    low_break_trade_uuid_list = low_break_trade_df['uuid'].to_list()
                    # UPDATE database
                    conn = postgres_client.pool.getconn()
                    curr = conn.cursor()
                    curr.execute(f"UPDATE trade SET trigger_switch = 0 WHERE uuid IN %s", (tuple(low_break_trade_uuid_list),))
                    conn.commit()
                    postgres_client.pool.putconn(conn)
                    self.low_break(low_break_trade_df)

                time.sleep(loop_interval_secs)
            except Exception as e:
                self.logger.error(f"Error in start_trigger_loop: {e}")
                # rollback the transaction if any error while inserting
                postgres_client.pool.putconn(conn, close=True)
                self.logger.error(f"Error in start_trigger_loop: {e}")
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"start_trigger_loop", content=traceback.format_exc(), code=None, sent_switch=0, send_counts=1, remark=None)
                time.sleep(10)

    def high_break(self, high_break_trade_df):
        for row_tup in high_break_trade_df.iterrows():
            def row_thread(row_tup):
                row = row_tup[1]
                user_uuid = row['user_uuid']
                telegram_id = self.user_info_dict[user_uuid]['telegram_id']
                telegram_name = self.user_info_dict[user_uuid]['telegram_name']
                alarm_num = self.user_info_dict[user_uuid]['alarm_num']
                alarm_period = self.user_info_dict[user_uuid]['alarm_period']
                trade_uuid = row['uuid']
                target_market_code = row['target_market_code']
                origin_market_code = row['origin_market_code']
                base_asset = row['base_asset']
                quote_asset = row['quote_asset']
                low = row['low']
                high = row['high']
                usdt_convertsion = row['usdt_conversion']
                

                ls_premium = row['LS_premium']
                ls_premium_value = row['LS_premium_value']
                sl_premium = row['SL_premium']
                sl_premium_value = row['SL_premium_value']

                print(f"프리미엄 상향돌파: {base_asset} {quote_asset} {sl_premium} {sl_premium_value}, 현재가격: {row['tp']}({round(row['scr'],2)})")
            row_thread = Thread(target=row_thread, args=(row_tup,), daemon=True)
            row_thread.start()

    def low_break(self, low_break_trade_df):
        for row_tup in low_break_trade_df.iterrows():
            def row_thread(row_tup):
                row = row_tup[1]
                user_uuid = row['user_uuid']
                telegram_id = self.user_info_dict[user_uuid]['telegram_id']
                telegram_name = self.user_info_dict[user_uuid]['telegram_name']
                alarm_num = self.user_info_dict[user_uuid]['alarm_num']
                alarm_period = self.user_info_dict[user_uuid]['alarm_period']
                trade_uuid = row['uuid']
                target_market_code = row['target_market_code']
                origin_market_code = row['origin_market_code']
                base_asset = row['base_asset']
                quote_asset = row['quote_asset']
                low = row['low']
                high = row['high']
                usdt_convertsion = row['usdt_conversion']
                

                ls_premium = row['LS_premium']
                ls_premium_value = row['LS_premium_value']
                sl_premium = row['SL_premium']
                sl_premium_value = row['SL_premium_value']

                print(f"프리미엄 하향돌파: {base_asset} {quote_asset} {ls_premium} {ls_premium_value}, 현재가격: {row['tp']}({round(row['scr'],2)})")
            row_thread = Thread(target=row_thread, args=(row_tup,), daemon=True)
            row_thread.start()

In [5]:
trigger = InitTrigger(admin_id, node, core.server_check_status_list, core.get_premium_df, enabled_markets, register_monitor_msg, core.remote_redis_client, core.db_dict, core.mongo_client, core.postgres_client, None)

2023-12-27 16:57:17,767 - root - INFO - user_info_df, exchange_config_df_dict, trade_df_dict have been initialized
2023-12-27 16:57:17,789 - root - INFO - start_trigger_loop: UPBIT_SPOT/KRW:BINANCE_USD_M/USDT


InitDBClient|SCHEMA: trade_core already exists.


2023-12-27 16:57:17,863 - root - INFO - postgres client has been initiated for UPBIT_SPOT/KRW:BINANCE_USD_M/USDT


프리미엄 하향돌파: BTC KRW 2.8156392120054083 2.8156392120054083, 현재가격: 57508000.0(0.83)


2023-12-27 16:57:29,522 - bravado.client - DEBUG - Market_info_all({'isDetails': True})
2023-12-27 16:57:29,535 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all?isDetails=true HTTP/1.1" 200 None
2023-12-27 16:57:29,892 - update_dollar - INFO - fetch_dollar|Dollar price (1297.0 KRW) has been updated.
2023-12-27 16:57:29,892 - update_dollar - INFO - fetch_dollar|Dollar price (1297.0 KRW) has been updated.
2023-12-27 16:57:30,102 - bravado.client - DEBUG - Market_info_all({})
2023-12-27 16:57:30,117 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all HTTP/1.1" 200 None
2023-12-27 16:57:30,122 - bravado.client - DEBUG - Trade_ticker({'markets': 'KRW-BTC,KRW-ETH,BTC-ETH,BTC-XRP,BTC-ETC,BTC-CVC,BTC-DGB,BTC-SC,BTC-SNT,BTC-WAVES,BTC-NMR,BTC-XEM,BTC-QTUM,BTC-BAT,BTC-LSK,BTC-STEEM,BTC-DOGE,BTC-BNT,BTC-XLM,BTC-ARDR,BTC-ARK,BTC-STORJ,BTC-GRS,BTC-RLC,USDT-BTC,USDT-ETH,USDT-XRP,USDT-ETC,KRW-NEO,KRW-MTL,KRW-XRP,KRW-ETC,KRW-SNT,KRW-WAVES,KRW

2023-12-27 16:57:59,570 - bravado.client - DEBUG - Market_info_all({'isDetails': True})
2023-12-27 16:57:59,580 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all?isDetails=true HTTP/1.1" 200 None
2023-12-27 16:58:00,286 - urllib3.connectionpool - DEBUG - https://fapi.binance.com:443 "GET /fapi/v1/exchangeInfo HTTP/1.1" 200 None
2023-12-27 16:58:00,350 - bravado.client - DEBUG - Market_info_all({})
2023-12-27 16:58:00,365 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all HTTP/1.1" 200 None
2023-12-27 16:58:00,370 - bravado.client - DEBUG - Trade_ticker({'markets': 'KRW-BTC,KRW-ETH,BTC-ETH,BTC-XRP,BTC-ETC,BTC-CVC,BTC-DGB,BTC-SC,BTC-SNT,BTC-WAVES,BTC-NMR,BTC-XEM,BTC-QTUM,BTC-BAT,BTC-LSK,BTC-STEEM,BTC-DOGE,BTC-BNT,BTC-XLM,BTC-ARDR,BTC-ARK,BTC-STORJ,BTC-GRS,BTC-RLC,USDT-BTC,USDT-ETH,USDT-XRP,USDT-ETC,KRW-NEO,KRW-MTL,KRW-XRP,KRW-ETC,KRW-SNT,KRW-WAVES,KRW-XEM,KRW-QTUM,KRW-LSK,KRW-STEEM,KRW-XLM,KRW-ARDR,KRW-ARK,KRW-STORJ,KRW-GRS,KRW

프리미엄 하향돌파: BTC KRW 2.7903736361087725 2.7903736361087725, 현재가격: 57508000.0(0.83)


2023-12-27 16:58:29,614 - bravado.client - DEBUG - Market_info_all({'isDetails': True})
2023-12-27 16:58:29,624 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all?isDetails=true HTTP/1.1" 200 None
2023-12-27 16:58:30,414 - urllib3.connectionpool - DEBUG - https://fapi.binance.com:443 "GET /fapi/v1/exchangeInfo HTTP/1.1" 200 None
2023-12-27 16:58:30,542 - bravado.client - DEBUG - Market_info_all({})
2023-12-27 16:58:30,553 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all HTTP/1.1" 200 None
2023-12-27 16:58:30,558 - bravado.client - DEBUG - Trade_ticker({'markets': 'KRW-BTC,KRW-ETH,BTC-ETH,BTC-XRP,BTC-ETC,BTC-CVC,BTC-DGB,BTC-SC,BTC-SNT,BTC-WAVES,BTC-NMR,BTC-XEM,BTC-QTUM,BTC-BAT,BTC-LSK,BTC-STEEM,BTC-DOGE,BTC-BNT,BTC-XLM,BTC-ARDR,BTC-ARK,BTC-STORJ,BTC-GRS,BTC-RLC,USDT-BTC,USDT-ETH,USDT-XRP,USDT-ETC,KRW-NEO,KRW-MTL,KRW-XRP,KRW-ETC,KRW-SNT,KRW-WAVES,KRW-XEM,KRW-QTUM,KRW-LSK,KRW-STEEM,KRW-XLM,KRW-ARDR,KRW-ARK,KRW-STORJ,KRW-GRS,KRW

프리미엄 하향돌파: BTC KRW 2.809799453681448 2.809799453681448, 현재가격: 57492000.0(0.81)


In [6]:
trigger.enabled_markets

['UPBIT_SPOT/KRW:BINANCE_USD_M/USDT']

In [6]:
trigger.trade_df_dict.values()

[]

2023-12-27 15:11:23,958 - bravado.client - DEBUG - Market_info_all({'isDetails': True})
2023-12-27 15:11:23,972 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all?isDetails=true HTTP/1.1" 200 None
2023-12-27 15:11:24,057 - update_dollar - INFO - fetch_dollar|Dollar price (1297.0 KRW) has been updated.
2023-12-27 15:11:24,057 - update_dollar - INFO - fetch_dollar|Dollar price (1297.0 KRW) has been updated.
2023-12-27 15:11:24,642 - bravado.client - DEBUG - Market_info_all({})
2023-12-27 15:11:24,654 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all HTTP/1.1" 200 None
2023-12-27 15:11:24,660 - urllib3.connectionpool - DEBUG - https://fapi.binance.com:443 "GET /fapi/v1/exchangeInfo HTTP/1.1" 200 None
2023-12-27 15:11:24,669 - bravado.client - DEBUG - Trade_ticker({'markets': 'KRW-BTC,KRW-ETH,BTC-ETH,BTC-XRP,BTC-ETC,BTC-CVC,BTC-DGB,BTC-SC,BTC-SNT,BTC-WAVES,BTC-NMR,BTC-XEM,BTC-QTUM,BTC-BAT,BTC-LSK,BTC-STEEM,BTC-DOGE,BTC-BNT,BTC-XL

In [ ]:
trigger.trade_df_dict[market_combi_code]

2023-12-27 15:04:30,650 - bravado.client - DEBUG - Market_info_all({'isDetails': True})


NameError: name 'market_combi_code' is not defined

2023-12-27 15:04:30,650 - bravado.client - DEBUG - Market_info_all({'isDetails': True})


2023-12-27 15:04:30,703 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all?isDetails=true HTTP/1.1" 200 None
2023-12-27 15:04:30,703 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all?isDetails=true HTTP/1.1" 200 None
2023-12-27 15:04:31,854 - urllib3.connectionpool - DEBUG - https://fapi.binance.com:443 "GET /fapi/v1/exchangeInfo HTTP/1.1" 200 None
2023-12-27 15:04:31,854 - urllib3.connectionpool - DEBUG - https://fapi.binance.com:443 "GET /fapi/v1/exchangeInfo HTTP/1.1" 200 None
2023-12-27 15:04:32,474 - bravado.client - DEBUG - Market_info_all({})
2023-12-27 15:04:32,474 - bravado.client - DEBUG - Market_info_all({})
2023-12-27 15:04:32,487 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all HTTP/1.1" 200 None
2023-12-27 15:04:32,487 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all HTTP/1.1" 200 None
2023-12-27 15:04:32,491 - bravado.client - DEBUG - Trade_ticker(

2023-12-27 10:08:11,898 - urllib3.connectionpool - DEBUG - https://fapi.binance.com:443 "GET /fapi/v1/ticker/24hr HTTP/1.1" 200 None
2023-12-27 10:08:13,292 - urllib3.connectionpool - DEBUG - https://api.binance.com:443 "GET /api/v3/exchangeInfo HTTP/1.1" 200 92924
2023-12-27 10:08:13,886 - urllib3.connectionpool - DEBUG - https://api.binance.com:443 "GET /api/v3/ticker/24hr HTTP/1.1" 200 144426
2023-12-27 10:08:15,630 - update_dollar - INFO - fetch_dollar|Dollar price (1295.0 KRW) has been updated.
2023-12-27 10:08:15,630 - update_dollar - INFO - fetch_dollar|Dollar price (1295.0 KRW) has been updated.
2023-12-27 10:08:19,662 - bravado.client - DEBUG - Market_info_all({})
2023-12-27 10:08:19,673 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all HTTP/1.1" 200 None
2023-12-27 10:08:19,676 - bravado.client - DEBUG - Trade_ticker({'markets': 'KRW-BTC,KRW-ETH,BTC-ETH,BTC-XRP,BTC-ETC,BTC-CVC,BTC-DGB,BTC-SC,BTC-SNT,BTC-WAVES,BTC-NMR,BTC-XEM,BTC-QTUM,BTC-BAT,BTC

In [10]:
trigger.get_premium_df(*market_combi_code.split(':'))

,tp_premium,LS_premium,SL_premium,LS_SL_spread,base_asset,quote_asset,tp,ap,bp,scr,converted_tp,converted_ap,converted_bp,dollar
0,3.214097,3.583684,3.214097,0.369586,1INCH,KRW,596.0,598.0,596.0,-3.089431,577.4405,577.4405,577.311,1295.0
1,3.269691,3.417114,3.259834,0.15728,AAVE,KRW,140100.0,140300.0,140100.0,1.521739,135664.2,135677.15,135664.2,1295.0
2,3.266077,3.427717,3.283128,0.144589,ADA,KRW,810.0,811.0,810.0,-0.613497,784.3815,784.252,784.1225,1295.0
3,3.229985,3.612604,3.229985,0.382618,ALGO,KRW,306.0,307.0,306.0,-2.857143,296.4255,296.4255,296.296,1295.0
4,3.286361,3.320824,3.028789,0.292035,ANKR,KRW,40.1,40.1,40.0,-1.231527,38.8241,38.8241,38.81115,1295.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,3.365458,4.01444,3.398024,0.616416,XLM,KRW,170.0,171.0,170.0,0.591716,164.465,164.4132,164.40025,1295.0
73,3.520116,3.660232,3.520116,0.140115,XRP,KRW,838.0,839.0,838.0,0.60024,809.5045,809.5045,809.375,1295.0
74,3.413356,3.509287,3.055525,0.453762,XTZ,KRW,1445.0,1445.0,1440.0,2.482269,1397.305,1397.305,1396.01,1295.0
75,2.827909,3.149021,2.827909,0.321111,ZIL,KRW,36.3,36.4,36.3,-0.274725,35.3017,35.3017,35.28875,1295.0


2023-12-27 08:42:44,006 - bravado.client - DEBUG - Market_info_all({'isDetails': True})
2023-12-27 08:42:44,018 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all?isDetails=true HTTP/1.1" 200 None
2023-12-27 08:42:45,099 - urllib3.connectionpool - DEBUG - https://fapi.binance.com:443 "GET /fapi/v1/exchangeInfo HTTP/1.1" 200 None
2023-12-27 08:42:45,590 - bravado.client - DEBUG - Market_info_all({})
2023-12-27 08:42:45,605 - urllib3.connectionpool - DEBUG - https://api.upbit.com:443 "GET /v1/market/all HTTP/1.1" 200 None
2023-12-27 08:42:45,609 - bravado.client - DEBUG - Trade_ticker({'markets': 'KRW-BTC,KRW-ETH,BTC-ETH,BTC-XRP,BTC-ETC,BTC-CVC,BTC-DGB,BTC-SC,BTC-SNT,BTC-WAVES,BTC-NMR,BTC-XEM,BTC-QTUM,BTC-BAT,BTC-LSK,BTC-STEEM,BTC-DOGE,BTC-BNT,BTC-XLM,BTC-ARDR,BTC-ARK,BTC-STORJ,BTC-GRS,BTC-RLC,USDT-BTC,USDT-ETH,USDT-XRP,USDT-ETC,KRW-NEO,KRW-MTL,KRW-XRP,KRW-ETC,KRW-SNT,KRW-WAVES,KRW-XEM,KRW-QTUM,KRW-LSK,KRW-STEEM,KRW-XLM,KRW-ARDR,KRW-ARK,KRW-STORJ,KRW-GRS,KRW

In [ ]:
conn = trigger.postgres_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

sql = """
SELECT *
FROM user_info
INNER JOIN exchange_config ON user_info.user_uuid = exchange_config.user_uuid;
"""

sql = """
SELECT * FROM user_info
"""


start = time.time()
curr.execute(sql)
result = curr.fetchall()
print(time.time() - start)
# temp_df = pd.DataFrame(result)

trigger.postgres_client.pool.putconn(conn)


In [ ]:
result[0]

In [ ]:
# Add addcoin manually

from uuid import uuid4

conn = trigger.postgres_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

sql = """
INSERT INTO trade (user_uuid, last_updated_datetime, uuid, base_asset, usdt_conversion, target_market_code, origin_market_code, low, high) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s);
"""
user_uuid = "test_uuid"
last_updated_datetime = datetime.datetime.utcnow()
uuid = uuid4().hex
base_asset = "BTC"
usdt_conversion = False
target_market_code = "UPBIT_SPOT/KRW"
origin_market_code = "BINANCE_USD_M/USDT"
low = 5
high = 6

curr.execute(sql, (user_uuid, last_updated_datetime, uuid, base_asset, usdt_conversion, target_market_code, origin_market_code, low, high))
conn.commit()

trigger.postgres_client.pool.putconn(conn)

In [ ]:
conn.rollback()
trigger.postgres_client.pool.putconn(conn)